[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/057.%20The%20Periodic%20Review%20%28Base-Stock%29%20Policy%20Problem/P57-Tier-4_executed.ipynb)

---


# 57. The Periodic Review (Base-Stock) Policy Problem

## Tier 4: Deep Reinforcement Learning with a Deep Q-Network

### Purpose and scope

This notebook simulates periodic-review inventory control and trains a Deep Q-Network (DQN) agent to choose order quantities. The environment includes uncertain, time-varying demand, replenishment lead time, holding costs, fixed ordering costs, and shortage costs. The notebook records training diagnostics, evaluates the learned policy, and describes the actions it selects in states encountered during simulation.

The action set is discrete: the DQN assigns a value to each candidate order ratio, and the chosen ratio determines an order quantity. This is a discrete-action DQN, not an optimizer for unrestricted continuous actions. Simulation advances in increments of one tenth of a week; inventory reviews occur at the specified review interval.

**Inputs** include demand and cost parameters, the review interval and lead time, and the DQN training settings. **Outputs** include training history, episode-level test statistics, diagnostic plots, and a policy analysis. The supplied traditional-policy figures are contextual reference values, not a controlled, matched evaluation in this notebook's simulated environment.

### Modeling assumptions

- Demand is nonnegative and is generated from a normal distribution whose mean and standard deviation vary over time. Negative draws are truncated at zero.
- Demand parameters are weekly quantities. Each simulation step represents \(0.1\) week: the mean is scaled by the step length and the standard deviation by its square root.
- Inventory is reviewed once per review interval. Orders are placed in a first-in, first-out (FIFO) pipeline and arrive after the modeled lead time.
- The observation includes on-hand inventory, total pipeline inventory, the position within the review interval, the individual pipeline entries in arrival order, and the ten most recent demand observations. The individual pipeline entries make the timing of future receipts observable. The demand trend and seasonal phase remain hidden, so the agent observes a partially observed process rather than a fully observed Markov state.
- Holding cost is charged per unit of inventory per week and scaled to the simulation step. A fixed ordering cost is charged when a positive order is placed. Shortage cost is charged per unit of unmet demand.
- Unmet demand is lost rather than backlogged. Inventory cannot fall below zero.
- Each episode lasts 52 simulated weeks. The model does not include capacity constraints, supplier disruptions, order minimums, backlogging, or a terminal salvage value.

### DQN formulation

A state (here, an observation) summarizes information available to the decision-maker; an action is a discrete order ratio; and a reward measures the immediate consequence of that decision. Reward is the negative of inventory cost, so maximizing accumulated reward corresponds to minimizing accumulated cost. Because demand trend and seasonal phase are hidden, the observation need not contain all information required for a fully Markov description. The DQN therefore learns an observation-conditioned action-value approximation from experience; this does not remove the partial observability.

Let \(o_t\) be the observation at time \(t\), \(a_t\) the selected discrete action, and \(r_t\) the reward. The one-step target used by DQN is

$$
y_t = r_t + \gamma (1-d_t) \max_{a'} Q_{\text{target}}(o_{t+1},a'),
$$

where \(d_t=1\) if the episode ends at this transition and \(d_t=0\) otherwise. The online network is trained to bring its estimate for the selected action toward this target. The target network supplies the next-observation estimate and is periodically synchronized with the online network. The replay buffer stores transitions; a randomly sampled minibatch is used to calculate a mean squared temporal-difference loss and one gradient update.

The agent uses an epsilon-greedy policy: with probability \(\epsilon\), it explores by choosing a random discrete action; otherwise, it chooses the action with the largest estimated Q-value. The discount factor \(\gamma\) controls the weight assigned to future rewards. The action grid, network architecture, replay buffer, target-network schedule, and training sequence are implemented below.

### What to examine

- Whether episode cost and service level change as the agent trains. These are noisy simulation measurements, not guarantees of convergence or optimality.
- The exploration rate \(\epsilon\), which decreases during training as the agent shifts from exploration toward greedy action selection.
- The distribution of actions in simulated observations. It describes the learned network's choices; it does not establish that those choices are optimal.
- The reference-policy comparison, interpreted cautiously because the supplied figures are not produced by a matched experiment here.

Deep reinforcement learning can represent complex observation-action relationships and adapt through experience. It also requires substantial computation, depends on training and modeling choices, may be unstable or overfit, and does not provide a general guarantee of optimality. Its suitability depends on the quality and quantity of available data, the value of adaptation, and the consequences of policy error.

In [ ]:
import random
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully.")

In [ ]:
class InventoryEnvironment:
    """Periodic-review inventory simulation with time-varying demand."""

    def __init__(
        self,
        demand_mean=12000,
        demand_std=2400,
        lead_time=0.5,
        holding_cost=0.15,
        ordering_cost=75,
        stockout_cost=8,
        max_inventory=50000,
        review_period=1.0,
        demand_volatility=0.1,
    ):
        self.demand_mean = demand_mean
        self.demand_std = demand_std
        self.lead_time = lead_time
        self.holding_cost = holding_cost
        self.ordering_cost = ordering_cost
        self.stockout_cost = stockout_cost
        self.max_inventory = max_inventory
        self.review_period = review_period
        self.demand_volatility = demand_volatility

        # Integer step counts keep review and receipt timing explicit.
        self.step_weeks = 0.1
        self.steps_per_week = 10
        self.review_steps = max(1, round(review_period / self.step_weeks))
        self.lead_steps = max(1, round(lead_time / self.step_weeks))
        self.episode_length = 52 * self.steps_per_week
        # Three core features, one feature per FIFO pipeline slot, and ten demands.
        self.state_dim = 3 + self.lead_steps + 10

        self.current_inventory = 0.0
        self.pipeline_orders = deque([0.0] * self.lead_steps)
        self.demand_history = deque(
            [demand_mean * self.step_weeks] * 10,
            maxlen=10,
        )
        self.time_step = 0
        self.episode_step = 0
        self.demand_trend = 0.0
        self.seasonal_phase = 0.0

        print("Inventory environment initialized:")
        print(f"  Weekly demand: N({self.demand_mean:,.0f}, {self.demand_std:,.0f}²)")
        print(f"  Lead time: {self.lead_time:g} weeks")
        print(f"  Review period: {self.review_period:g} weeks")
        print(f"  Simulation step: {self.step_weeks:g} week")
        print(f"  State dimension: {self.state_dim}")

    def reset(self):
        """Start an episode with stock covering the review and lead times."""
        self.current_inventory = self.demand_mean * (
            self.review_period + self.lead_time
        )
        self.pipeline_orders = deque([0.0] * self.lead_steps)
        self.demand_history = deque(
            [self.demand_mean * self.step_weeks] * 10,
            maxlen=10,
        )
        self.time_step = 0
        self.episode_step = 0
        self.demand_trend = random.uniform(-0.02, 0.02)
        self.seasonal_phase = random.uniform(0, 2 * np.pi)
        return self._get_state()

    def _get_state(self):
        """Return normalized inventory, pipeline schedule, review phase, and demand history."""
        norm_inventory = self.current_inventory / self.max_inventory
        norm_pipeline = sum(self.pipeline_orders) / self.max_inventory
        time_fraction = (
            (self.episode_step % self.review_steps) / self.review_steps
        )
        norm_pipeline_schedule = [
            order / self.max_inventory for order in self.pipeline_orders
        ]

        average_demand = np.mean(self.demand_history)
        if average_demand > 0:
            norm_demand_history = [
                demand / (2 * average_demand)
                for demand in self.demand_history
            ]
        else:
            norm_demand_history = [0.0] * len(self.demand_history)

        state = [norm_inventory, norm_pipeline, time_fraction]
        state.extend(norm_pipeline_schedule)
        state.extend(norm_demand_history)
        return np.asarray(state, dtype=np.float32)

    def _generate_demand(self):
        """Generate nonnegative demand with trend, seasonality, and volatility."""
        annual_phase = 2 * np.pi * self.episode_step / self.episode_length
        trend_effect = 1 + self.demand_trend * self.episode_step / self.episode_length
        seasonal_effect = 1 + 0.1 * np.sin(self.seasonal_phase + annual_phase)
        current_weekly_mean = (
            self.demand_mean * trend_effect * seasonal_effect
        )

        volatility_phase = 2 * np.pi * self.episode_step / (self.episode_length / 4)
        volatility_factor = 1 + self.demand_volatility * np.sin(volatility_phase)
        step_mean = current_weekly_mean * self.step_weeks
        step_std = (
            self.demand_std
            * volatility_factor
            * np.sqrt(self.step_weeks)
        )
        return max(0.0, np.random.normal(step_mean, step_std))

    def step(self, action):
        """Advance one simulation step using an order ratio in the interval [0, 1]."""
        action = float(np.clip(action, 0.0, 1.0))
        max_order = self.max_inventory * 0.5
        order_quantity = 0.0

        # The oldest entry arrives now; FIFO positions encode remaining lead time.
        received_order = self.pipeline_orders.popleft()
        self.current_inventory += received_order

        demand = self._generate_demand()
        self.demand_history.append(demand)
        shortage = max(0.0, demand - self.current_inventory)
        self.current_inventory = max(0.0, self.current_inventory - demand)

        is_review_step = self.episode_step % self.review_steps == 0
        if is_review_step:
            order_quantity = action * max_order
        self.pipeline_orders.append(order_quantity)

        holding_cost = (
            self.holding_cost * self.current_inventory * self.step_weeks
        )
        ordering_cost = self.ordering_cost if order_quantity > 0 else 0.0
        stockout_cost = self.stockout_cost * shortage
        total_cost = holding_cost + ordering_cost + stockout_cost

        self.episode_step += 1
        self.time_step += 1
        done = self.episode_step >= self.episode_length
        reward = -total_cost

        info = {
            "total_cost": total_cost,
            "holding_cost": holding_cost,
            "ordering_cost": ordering_cost,
            "stockout_cost": stockout_cost,
            "demand": demand,
            "shortage": shortage,
            "order_quantity": order_quantity,
            "inventory_level": self.current_inventory,
        }
        return self._get_state(), reward, done, info

print("InventoryEnvironment class defined successfully.")

In [ ]:
class DQNNetwork:
    """Fully connected neural network that approximates action values."""

    def __init__(self, state_dim, action_dim, hidden_dim=64, learning_rate=0.001):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.learning_rate = learning_rate

        self.weights = {
            "W0": np.random.randn(state_dim, hidden_dim) * 0.1,
            "W1": np.random.randn(hidden_dim, hidden_dim) * 0.1,
            "W2": np.random.randn(hidden_dim, action_dim) * 0.1,
        }
        self.biases = {
            "b0": np.zeros(hidden_dim),
            "b1": np.zeros(hidden_dim),
            "b2": np.zeros(action_dim),
        }
        self.grad_weights = {
            key: np.zeros_like(value) for key, value in self.weights.items()
        }
        self.grad_biases = {
            key: np.zeros_like(value) for key, value in self.biases.items()
        }

        print("DQN network initialized:")
        print(f"  State dimension: {state_dim}")
        print(f"  Action dimension: {action_dim}")
        print(f"  Hidden units per layer: {hidden_dim}")
        print(f"  Learning rate: {learning_rate}")

    @staticmethod
    def relu(values):
        return np.maximum(0, values)

    @staticmethod
    def relu_derivative(values):
        return (values > 0).astype(float)

    def forward(self, state):
        """Compute one Q-value for each discrete action."""
        self.cache = {}
        z0 = np.dot(state, self.weights["W0"]) + self.biases["b0"]
        a0 = self.relu(z0)
        z1 = np.dot(a0, self.weights["W1"]) + self.biases["b1"]
        a1 = self.relu(z1)
        q_values = np.dot(a1, self.weights["W2"]) + self.biases["b2"]

        self.cache.update({"z0": z0, "a0": a0, "z1": z1, "a1": a1})
        return q_values

    def forward_batch(self, states):
        """Compute action values for a minibatch using array operations."""
        z0 = np.dot(states, self.weights["W0"]) + self.biases["b0"]
        a0 = self.relu(z0)
        z1 = np.dot(a0, self.weights["W1"]) + self.biases["b1"]
        a1 = self.relu(z1)
        q_values = np.dot(a1, self.weights["W2"]) + self.biases["b2"]
        return q_values, (z0, a0, z1, a1)

    def backward(self, state, target_q_values):
        """Compute gradients of the squared-error loss for one state."""
        q_values = self.forward(state)
        loss_gradient = 2 * (q_values - target_q_values)

        dz2 = loss_gradient
        self.grad_weights["W2"] = np.outer(self.cache["a1"], dz2)
        self.grad_biases["b2"] = dz2

        da1 = np.dot(dz2, self.weights["W2"].T)
        dz1 = da1 * self.relu_derivative(self.cache["z1"])
        self.grad_weights["W1"] = np.outer(self.cache["a0"], dz1)
        self.grad_biases["b1"] = dz1

        da0 = np.dot(dz1, self.weights["W1"].T)
        dz0 = da0 * self.relu_derivative(self.cache["z0"])
        self.grad_weights["W0"] = np.outer(state, dz0)
        self.grad_biases["b0"] = dz0

    def backward_batch(self, states, target_q_values):
        """Compute mean squared-error gradients across a sampled minibatch."""
        q_values, cache = self.forward_batch(states)
        z0, a0, z1, a1 = cache
        batch_size = states.shape[0]
        dz2 = 2 * (q_values - target_q_values) / batch_size

        self.grad_weights["W2"] = np.dot(a1.T, dz2)
        self.grad_biases["b2"] = np.sum(dz2, axis=0)

        da1 = np.dot(dz2, self.weights["W2"].T)
        dz1 = da1 * self.relu_derivative(z1)
        self.grad_weights["W1"] = np.dot(a0.T, dz1)
        self.grad_biases["b1"] = np.sum(dz1, axis=0)

        da0 = np.dot(dz1, self.weights["W1"].T)
        dz0 = da0 * self.relu_derivative(z0)
        self.grad_weights["W0"] = np.dot(states.T, dz0)
        self.grad_biases["b0"] = np.sum(dz0, axis=0)

    def update_weights(self):
        """Apply one gradient-descent update to weights and their matching biases."""
        for weight_key, bias_key in (("W0", "b0"), ("W1", "b1"), ("W2", "b2")):
            self.weights[weight_key] -= (
                self.learning_rate * self.grad_weights[weight_key]
            )
            self.biases[bias_key] -= (
                self.learning_rate * self.grad_biases[bias_key]
            )

    def get_action(self, state, epsilon=0.0):
        """Select a discrete action using an epsilon-greedy rule."""
        if np.random.random() < epsilon:
            return np.random.randint(0, self.action_dim)
        return int(np.argmax(self.forward(state)))

    def copy_weights_from(self, other_network):
        """Copy parameters so the target network matches the online network."""
        for key in self.weights:
            self.weights[key] = other_network.weights[key].copy()
        for key in self.biases:
            self.biases[key] = other_network.biases[key].copy()

print("DQNNetwork class defined successfully.")

In [ ]:
class DQNAgent:
    """DQN agent with epsilon-greedy exploration, replay, and a target network."""

    def __init__(
        self,
        state_dim,
        action_dim=20,
        learning_rate=0.001,
        gamma=0.99,
        epsilon=1.0,
        epsilon_decay=0.995,
        epsilon_min=0.05,
        memory_size=10000,
    ):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.batch_size = 32

        self.q_network = DQNNetwork(
            state_dim,
            action_dim,
            learning_rate=learning_rate,
        )
        self.target_network = DQNNetwork(
            state_dim,
            action_dim,
            learning_rate=learning_rate,
        )
        self.update_target_network()
        self.memory = deque(maxlen=memory_size)
        self.training_step = 0

        print("DQN agent initialized:")
        print(f"  State dimension: {state_dim}; action dimension: {action_dim}")
        print(f"  Learning rate: {learning_rate}; discount factor: {gamma}")
        print(f"  Epsilon: {epsilon} to {epsilon_min} (decay: {epsilon_decay})")
        print(f"  Replay capacity: {memory_size}; batch size: {self.batch_size}")

    def update_target_network(self):
        """Synchronize the target network with the online network."""
        self.target_network.copy_weights_from(self.q_network)

    def remember(self, state, action, reward, next_state, done):
        """Store a transition for later replay."""
        self.memory.append(
            (state.copy(), action, reward, next_state.copy(), done)
        )

    def act(self, state, training=True):
        """Choose an action index and map it to an order ratio in [0, 1]."""
        if training and random.random() < self.epsilon:
            action_index = random.randrange(self.action_dim)
        else:
            action_index = int(np.argmax(self.q_network.forward(state)))
        return action_index / (self.action_dim - 1)

    def replay(self):
        """Train once on a random minibatch of stored transitions."""
        if len(self.memory) < self.batch_size:
            return

        batch = random.sample(self.memory, self.batch_size)
        states = np.asarray([item[0] for item in batch], dtype=np.float32)
        actions = np.asarray([item[1] for item in batch], dtype=np.float32)
        rewards = np.asarray([item[2] for item in batch], dtype=np.float32)
        next_states = np.asarray([item[3] for item in batch], dtype=np.float32)
        dones = np.asarray([item[4] for item in batch], dtype=bool)

        action_indices = np.rint(actions * (self.action_dim - 1)).astype(int)
        action_indices = np.clip(action_indices, 0, self.action_dim - 1)

        next_q_values = self.target_network.forward_batch(next_states)[0]
        target_values = rewards + self.gamma * (~dones) * np.max(
            next_q_values,
            axis=1,
        )

        target_q_values = self.q_network.forward_batch(states)[0].copy()
        target_q_values[np.arange(self.batch_size), action_indices] = target_values
        self.q_network.backward_batch(states, target_q_values)
        self.q_network.update_weights()

        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

        self.training_step += 1
        if self.training_step % 100 == 0:
            self.update_target_network()

print("DQNAgent class defined successfully.")

In [ ]:
env = InventoryEnvironment(
    demand_mean=12000,
    demand_std=2400,
    lead_time=0.5,
    holding_cost=0.15,
    ordering_cost=75,
    stockout_cost=8,
    max_inventory=50000,
    review_period=1.0,
    demand_volatility=0.1,
)

agent = DQNAgent(
    state_dim=env.state_dim,
    action_dim=20,
    learning_rate=0.001,
    gamma=0.99,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.05,
    memory_size=10000,
)

print("Environment and agent initialized successfully.")

In [ ]:
def train_agent(agent, env, num_episodes=1000, max_steps_per_episode=520, verbose=True):
    """Train the DQN and return one record of simulation metrics per episode."""
    training_history = []
    print(f"Starting training for {num_episodes} episodes.")
    print("-" * 60)

    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0.0
        total_cost = 0.0
        total_shortage = 0.0
        total_demand = 0.0
        inventory_levels = []
        steps = 0
        info = {"inventory_level": env.current_inventory}

        while steps < max_steps_per_episode:
            action = agent.act(state, training=True)
            next_state, reward, done, info = env.step(action)
            agent.remember(state, action, reward, next_state, done)

            total_reward += reward
            total_cost += info["total_cost"]
            total_shortage += info["shortage"]
            total_demand += info["demand"]
            inventory_levels.append(info["inventory_level"])

            if len(agent.memory) >= agent.batch_size:
                agent.replay()

            state = next_state
            steps += 1
            if done:
                break

        elapsed_weeks = steps * env.step_weeks
        average_weekly_cost = total_cost / elapsed_weeks if elapsed_weeks else 0.0
        service_level = (
            1 - total_shortage / total_demand if total_demand > 0 else 0.0
        )
        average_inventory = (
            float(np.mean(inventory_levels)) if inventory_levels else 0.0
        )

        training_history.append(
            {
                "episode": episode + 1,
                "total_reward": total_reward,
                "avg_cost": average_weekly_cost,
                "service_level": service_level,
                "avg_inventory": average_inventory,
                "epsilon": agent.epsilon,
                "steps": steps,
            }
        )

        if verbose and (episode + 1) % 100 == 0:
            print(
                f"Episode {episode + 1:4d}: "
                f"Reward = {total_reward:10.0f}, "
                f"Cost = ${average_weekly_cost:8.2f}/week, "
                f"Service = {service_level:6.3f}, "
                f"Epsilon = {agent.epsilon:5.3f}"
            )

    print("-" * 60)
    print("Training completed.")
    return training_history


def test_agent(agent, env, num_episodes=10, max_steps_per_episode=520):
    """Evaluate a greedy policy and return episode-level test metrics."""
    test_results = []
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0
    print(f"\nTesting the greedy policy for {num_episodes} episodes.")

    try:
        for episode in range(num_episodes):
            state = env.reset()
            total_reward = 0.0
            total_cost = 0.0
            total_shortage = 0.0
            total_demand = 0.0
            inventory_levels = []
            order_quantities = []
            steps = 0
            done = False

            while steps < max_steps_per_episode and not done:
                action = agent.act(state, training=False)
                next_state, reward, done, info = env.step(action)
                total_reward += reward
                total_cost += info["total_cost"]
                total_shortage += info["shortage"]
                total_demand += info["demand"]
                inventory_levels.append(info["inventory_level"])
                if info["order_quantity"] > 0:
                    order_quantities.append(info["order_quantity"])
                state = next_state
                steps += 1

            elapsed_weeks = steps * env.step_weeks
            average_weekly_cost = total_cost / elapsed_weeks if elapsed_weeks else 0.0
            service_level = (
                1 - total_shortage / total_demand if total_demand > 0 else 0.0
            )
            test_results.append(
                {
                    "episode": episode + 1,
                    "avg_cost": average_weekly_cost,
                    "service_level": service_level,
                    "avg_inventory": (
                        float(np.mean(inventory_levels)) if inventory_levels else 0.0
                    ),
                    "total_orders": len(order_quantities),
                    "avg_order_size": (
                        float(np.mean(order_quantities)) if order_quantities else 0.0
                    ),
                    "total_reward": total_reward,
                }
            )
    finally:
        agent.epsilon = original_epsilon

    return test_results

print("Training and testing functions defined successfully.")

## Train the agent

Each episode simulates up to 52 weeks. At each simulation step, the agent selects an action; the environment places an order only when a review occurs, so actions between reviews do not create orders. During training, the agent explores with an epsilon-greedy policy, stores transitions in replay memory, and updates the online network from randomly sampled minibatches. A minibatch is processed with array operations and contributes one gradient update, reducing repeated per-transition computation while retaining the replay-based DQN training sequence. The target network is synchronized periodically. The training run is computationally intensive because it contains many simulation steps and replay updates.

In [ ]:
training_history = train_agent(agent, env, num_episodes=500, verbose=True)
training_df = pd.DataFrame(training_history)

assert not training_df.empty
assert training_df["steps"].between(1, env.episode_length).all()
print(f"Recorded metrics for {len(training_df)} training episodes.")

## Evaluate the greedy policy

Evaluation disables exploration but continues to simulate uncertain demand and randomized demand patterns. Episode cost is reported in currency per simulated week. The service level is the fraction of total demand met, calculated as \(1-\frac{\text{total shortage}}{\text{total demand}}\). These metrics summarize the tested episodes; they are not guarantees about future performance.

In [ ]:
test_results = test_agent(agent, env, num_episodes=10)
test_df = pd.DataFrame(test_results)

if test_df.empty:
    raise RuntimeError("Testing produced no episode results.")

test_stats = {
    "avg_cost": test_df["avg_cost"].mean(),
    "cost_std": test_df["avg_cost"].std(),
    "avg_service": test_df["service_level"].mean(),
    "service_std": test_df["service_level"].std(),
    "avg_inventory": test_df["avg_inventory"].mean(),
    "inventory_std": test_df["avg_inventory"].std(),
    "avg_orders": test_df["total_orders"].mean(),
    "avg_order_size": test_df["avg_order_size"].mean(),
}

print("TEST RESULTS SUMMARY")
print("=" * 60)
print(
    f"Average cost: ${test_stats['avg_cost']:.2f} "
    f"± ${test_stats['cost_std']:.2f} per week"
)
print(
    f"Average service level: {test_stats['avg_service']:.3f} "
    f"± {test_stats['service_std']:.3f}"
)
print(
    f"Average inventory: {test_stats['avg_inventory']:.0f} "
    f"± {test_stats['inventory_std']:.0f} units"
)
print(f"Average orders per episode: {test_stats['avg_orders']:.1f}")
print(f"Average positive order size: {test_stats['avg_order_size']:.0f} units")

# These supplied Tier 2 figures are contextual references, not a matched
# rerun of the traditional policy under identical simulation conditions.
traditional_cost = 2284.73
traditional_service = 0.942

print("\nCONTEXTUAL REFERENCE COMPARISON")
print("=" * 60)
print(f"Traditional reference cost: ${traditional_cost:.2f} per reported period")
print(f"Traditional reference service level: {traditional_service:.3f}")
print(f"DQN test cost: ${test_stats['avg_cost']:.2f} per simulated week")
print(f"DQN test service level: {test_stats['avg_service']:.3f}")
print(
    "These figures are descriptive context only; the reference period and "
    "simulation conditions are not matched to the DQN test episodes."
)

## Training and performance diagnostics

The first three panels show episode-level cost, service level, and exploration rate. Moving averages help reveal broad patterns while raw observations retain the variability of individual episodes. The final panel places the supplied reference values beside the DQN test summaries after separate normalization. It is a visual aid only: the reference cost is reported per period, whereas the DQN cost is reported per simulated week, and the experiments are not matched. Numerical labels show the underlying values. Cost and service also use different units and directions of preference, so examine each metric separately.

In [ ]:
window_size = 50
training_df["cost_ma"] = training_df["avg_cost"].rolling(
    window=window_size,
    min_periods=1,
).mean()
training_df["service_ma"] = training_df["service_level"].rolling(
    window=window_size,
    min_periods=1,
).mean()

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Deep Reinforcement Learning for Periodic-Review Inventory Control",
    fontsize=16,
    fontweight="bold",
)

ax1.plot(
    training_df["episode"],
    training_df["avg_cost"],
    color="tab:blue",
    alpha=0.3,
    linewidth=0.8,
    label="Episode cost",
)
ax1.plot(
    training_df["episode"],
    training_df["cost_ma"],
    color="tab:blue",
    linewidth=2,
    label=f"{window_size}-episode moving average",
)
ax1.set_xlabel("Training episode")
ax1.set_ylabel("Cost (currency per simulated week)")
ax1.set_title("Training cost")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(
    training_df["episode"],
    training_df["service_level"],
    color="tab:green",
    alpha=0.3,
    linewidth=0.8,
    label="Episode service level",
)
ax2.plot(
    training_df["episode"],
    training_df["service_ma"],
    color="tab:green",
    linewidth=2,
    label=f"{window_size}-episode moving average",
)
ax2.set_xlabel("Training episode")
ax2.set_ylabel("Service level (fraction of demand met)")
ax2.set_title("Training service level")
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3.plot(
    training_df["episode"],
    training_df["epsilon"],
    color="tab:orange",
    linewidth=2,
)
ax3.set_xlabel("Training episode")
ax3.set_ylabel("Epsilon (exploration probability)")
ax3.set_title("Exploration rate")
ax3.grid(True, alpha=0.3)

methods = ["Traditional reference", "DQN test"]
costs = [traditional_cost, test_stats["avg_cost"]]
services = [traditional_service, test_stats["avg_service"]]
x_values = np.arange(len(methods))
bar_width = 0.35
cost_scale = max(costs) if max(costs) > 0 else 1.0
service_scale = max(services) if max(services) > 0 else 1.0
cost_normalized = [value / cost_scale for value in costs]
service_normalized = [value / service_scale for value in services]

ax4.bar(
    x_values - bar_width / 2,
    cost_normalized,
    bar_width,
    label="Normalized cost (lower is preferable)",
    alpha=0.8,
)
ax4.bar(
    x_values + bar_width / 2,
    service_normalized,
    bar_width,
    label="Normalized service (higher is preferable)",
    alpha=0.8,
)
ax4.set_xlabel("Policy")
ax4.set_ylabel("Normalized value")
ax4.set_title("Contextual reference comparison")
ax4.set_xticks(x_values)
ax4.set_xticklabels(methods, rotation=10)
ax4.set_ylim(0, 1.15)
ax4.legend(fontsize=8)
ax4.grid(True, axis="y", alpha=0.3)

for index, (cost, service) in enumerate(zip(costs, services)):
    ax4.text(
        index - bar_width / 2,
        cost_normalized[index] + 0.02,
        f"${cost:.0f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )
    ax4.text(
        index + bar_width / 2,
        service_normalized[index] + 0.02,
        f"{service:.3f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

fig.tight_layout()
plt.show()
print(
    "Interpret the learning curves alongside their episode-to-episode "
    "variation; a moving average alone does not establish convergence."
)

## Policy analysis

A policy is a rule that maps an observed state to an action. The analysis below records observations encountered while the greedy agent interacts with the environment, then groups selected order ratios by on-hand inventory and position within the review interval. An order ratio of \(0\) means no order; a ratio of \(1\) means the maximum permitted order. These group summaries describe behavior in the sampled observations and should be interpreted alongside their sample counts.

In [ ]:
def analyze_policy(agent, env, num_test_states=100):
    """Collect greedy actions and Q-values from observations encountered in rollouts."""
    policy_data = []
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0

    try:
        while len(policy_data) < num_test_states:
            state = env.reset()
            done = False

            while not done and len(policy_data) < num_test_states:
                q_values = agent.q_network.forward(state)
                best_action_index = int(np.argmax(q_values))
                action_ratio = best_action_index / (agent.action_dim - 1)

                policy_data.append(
                    {
                        "inventory_level": state[0] * env.max_inventory,
                        "pipeline_level": state[1] * env.max_inventory,
                        "time_fraction": state[2],
                        "action": action_ratio,
                        "q_value": q_values[best_action_index],
                        "q_variance": np.var(q_values),
                    }
                )

                state, _, done, _ = env.step(action_ratio)
    finally:
        agent.epsilon = original_epsilon

    return pd.DataFrame(policy_data)


def describe_action_group(data, label):
    """Print a group's mean action and sample size, including empty groups."""
    if data.empty:
        print(f"{label}: no sampled states")
    else:
        print(
            f"{label}: mean order ratio = {data['action'].mean():.3f} "
            f"({len(data)} states)"
        )


policy_df = analyze_policy(agent, env, num_test_states=200)
print(f"Policy analysis: {len(policy_df)} sampled states")
print(f"Mean order ratio: {policy_df['action'].mean():.3f}")
print(f"Order-ratio standard deviation: {policy_df['action'].std():.3f}")
print(f"Mean selected-action Q-value: {policy_df['q_value'].mean():.2f}")
print(f"Mean variance across action Q-values: {policy_df['q_variance'].mean():.2f}")

low_inventory = policy_df[policy_df["inventory_level"] < 15000]
medium_inventory = policy_df[
    (policy_df["inventory_level"] >= 15000)
    & (policy_df["inventory_level"] < 25000)
]
high_inventory = policy_df[policy_df["inventory_level"] >= 25000]

print("\nAction patterns by on-hand inventory:")
describe_action_group(low_inventory, "Low inventory (< 15,000 units)")
describe_action_group(medium_inventory, "Medium inventory (15,000–25,000 units)")
describe_action_group(high_inventory, "High inventory (>= 25,000 units)")

print("\nAction patterns by position within the review interval:")
early_time = policy_df[policy_df["time_fraction"] < 0.3]
mid_time = policy_df[
    (policy_df["time_fraction"] >= 0.3)
    & (policy_df["time_fraction"] < 0.7)
]
late_time = policy_df[policy_df["time_fraction"] >= 0.7]
describe_action_group(early_time, "Early (< 30%)")
describe_action_group(mid_time, "Middle (30%–70%)")
describe_action_group(late_time, "Late (>= 70%)")

## Summary and interpretation

This notebook represents periodic-review inventory control as a partially observed sequential decision problem and applies a DQN with a replay buffer and a periodically synchronized target network. The agent's observation includes on-hand inventory, total pipeline inventory, the FIFO pipeline schedule, review timing, and recent demand. Hidden trend and seasonal variables mean that this observation is not a complete Markov state. The discrete action grid represents feasible order ratios, and reward is the negative of holding, ordering, and shortage costs.

Use the test episodes to assess the policy under the simulated demand process, and use the training plots to inspect learning behavior and exploration. Policy-group averages are descriptive and depend on which observations the learned policy visits. The traditional figures are contextual references, not evidence of a controlled head-to-head comparison. A lower observed cost or higher observed service level in this run does not establish that the DQN is optimal or that the result will generalize to other demand processes.

The model's scope is deliberately limited: it uses a specified demand generator and simplified replenishment and shortage rules. Real applications would require validating those assumptions and accounting for operational constraints and data quality. Deep reinforcement learning is most relevant when adaptation to complex or changing conditions is valuable and sufficient computational resources and evaluation data are available; its training cost, sensitivity, limited interpretability, and lack of general optimality guarantees must also be considered.